In [1]:
%pip install -qU langchain-text-splitters
from langchain_text_splitters import (
    Language,
    RecursiveCharacterTextSplitter,
)


[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:

PYTHON_CODE = """
def hello_world():
    print("Hello, World!")

# Call the function
hello_world()
"""
python_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON, chunk_size=50, chunk_overlap=0
)
python_docs = python_splitter.create_documents([PYTHON_CODE])
python_docs

[Document(metadata={}, page_content='def hello_world():\n    print("Hello, World!")'),
 Document(metadata={}, page_content='# Call the function\nhello_world()')]

In [3]:
COBOL_CODE="""
      ******************************************************************
      * Created: Thu, 14 Mar 2024 23:37:26 GMT
      * Generated by: IBM watsonx Code Assistant for Z Refactoring
      * Assistant
      * Workbook name: LGICDB01
      * Workbook id: 749a805c-1f53-4f65-b1a1-be464cca3935
      * Project: $GenApp_9c635951-ff98-4c2e-b721-5163f063503d
      ******************************************************************
  
       IDENTIFICATION DIVISION.
       PROGRAM-ID. INSCUST.

       DATA DIVISION. 
       WORKING-STORAGE SECTION.
       01  CA-ERROR-MSG.
           03 FILLER                   PIC X(9)  VALUE 'COMMAREA='.
           03 CA-DATA                  PIC X(90) VALUE SPACES.
       01  DB2-OUT-INTEGERS.
           03 DB2-CUSTOMERNUM-INT   PIC S9(9) COMP.
       77 LGAC-NCS                     PIC X(2)  VALUE 'ON'.
       01  WS-ABSTIME                  PIC S9(8) COMP VALUE +0.
       01  WS-TIME                     PIC X(8)  VALUE SPACES.
       01  WS-DATE                     PIC X(10) VALUE SPACES.
       01  DFHCOMMAREA-1.
           COPY LGCMAREA.
       01  ERROR-MSG.
           03 EM-VARIABLE.
             05 FILLER                 PIC X(6)  VALUE ' CNUM='.
             05 EM-CUSNUM              PIC X(10)  VALUE SPACES.
             05 FILLER                 PIC X(6)  VALUE ' PNUM='.
             05 EM-POLNUM              PIC X(10)  VALUE SPACES.
             05 EM-SQLREQ              PIC X(16) VALUE SPACES.
             05 FILLER                 PIC X(9)  VALUE ' SQLCODE='.
             05 EM-SQLRC               PIC +9(5) USAGE DISPLAY.
           03 EM-DATE                  PIC X(8)  VALUE SPACES.
           03 FILLER                   PIC X     VALUE SPACES.
           03 EM-TIME                  PIC X(6)  VALUE SPACES.
           03 FILLER                   PIC X(9)  VALUE ' LGACDB01'.
        COPY SQLCA.

      *     EXEC SQL
      *       INCLUDE SQLCA
      *     END-EXEC.

       LINKAGE SECTION.

       PROCEDURE DIVISION.
       INSERT-CUSTOMER.
      *================================================================*
      * Insert row into Customer table based on customer number        *
      *================================================================*
           MOVE ' INSERT CUSTOMER' TO EM-SQLREQ
      *================================================================*
           IF LGAC-NCS = 'ON'
             EXEC SQL
               INSERT INTO CUSTOMER
                         ( CUSTOMERNUMBER,
                           FIRSTNAME,
                           LASTNAME,
                           DATEOFBIRTH,
                           HOUSENAME,
                           HOUSENUMBER,
                           POSTCODE,
                           PHONEMOBILE,
                           PHONEHOME,
                           EMAILADDRESS )
                  VALUES ( :DB2-CUSTOMERNUM-INT,
                           :CA-FIRST-NAME,
                           :CA-LAST-NAME,
                           :CA-DOB,
                           :CA-HOUSE-NAME,
                           :CA-HOUSE-NUM,
                           :CA-POSTCODE,
                           :CA-PHONE-MOBILE,
                           :CA-PHONE-HOME,
                           :CA-EMAIL-ADDRESS )
             END-EXEC
             IF SQLCODE NOT EQUAL 0
               MOVE '90' TO CA-RETURN-CODE
               PERFORM WRITE-ERROR-MESSAGE
               EXEC CICS RETURN END-EXEC
             END-IF
           ELSE
             EXEC SQL
               INSERT INTO CUSTOMER
                         ( CUSTOMERNUMBER,
                           FIRSTNAME,
                           LASTNAME,
                           DATEOFBIRTH,
                           HOUSENAME,
                           HOUSENUMBER,
                           POSTCODE,
                           PHONEMOBILE,
                           PHONEHOME,
                           EMAILADDRESS )
                  VALUES ( DEFAULT,
                           :CA-FIRST-NAME,
                           :CA-LAST-NAME,
                           :CA-DOB,
                           :CA-HOUSE-NAME,
                           :CA-HOUSE-NUM,
                           :CA-POSTCODE,
                           :CA-PHONE-MOBILE,
                           :CA-PHONE-HOME,
                           :CA-EMAIL-ADDRESS )
             END-EXEC
             IF SQLCODE NOT EQUAL 0
               MOVE '90' TO CA-RETURN-CODE
               PERFORM WRITE-ERROR-MESSAGE
               EXEC CICS RETURN END-EXEC
             END-IF
      *    get value of assigned customer number
               EXEC SQL
                 SET :DB2-CUSTOMERNUM-INT = IDENTITY_VAL_LOCAL()
               END-EXEC
           END-IF.

           MOVE DB2-CUSTOMERNUM-INT TO CA-CUSTOMER-NUM.

           EXIT.

       WRITE-ERROR-MESSAGE.
      * Save SQLCODE in message
           MOVE SQLCODE TO EM-SQLRC
      * Obtain and format current time and date
           EXEC CICS ASKTIME ABSTIME(WS-ABSTIME)
           END-EXEC
           EXEC CICS FORMATTIME ABSTIME(WS-ABSTIME)
                     MMDDYYYY(WS-DATE)
                     TIME(WS-TIME)
           END-EXEC
           MOVE WS-DATE TO EM-DATE
           MOVE WS-TIME TO EM-TIME
      * Write output message to TDQ
           EXEC CICS LINK PROGRAM('LGSTSQ')
                     COMMAREA(ERROR-MSG)
                     LENGTH(LENGTH OF ERROR-MSG)
           END-EXEC.
      * Write 90 bytes or as much as we have of commarea to TDQ
           IF EIBCALEN > 0 THEN
             IF EIBCALEN < 91 THEN
               MOVE DFHCOMMAREA-1(1:EIBCALEN) TO CA-DATA
               EXEC CICS LINK PROGRAM('LGSTSQ')
                         COMMAREA(CA-ERROR-MSG)
                         LENGTH(LENGTH OF CA-ERROR-MSG)
               END-EXEC
             ELSE
               MOVE DFHCOMMAREA-1(1:90) TO CA-DATA
               EXEC CICS LINK PROGRAM('LGSTSQ')
                         COMMAREA(CA-ERROR-MSG)
                         LENGTH(LENGTH OF CA-ERROR-MSG)
               END-EXEC
             END-IF
           END-IF.
           EXIT.

           EXIT PROGRAM.
"""


In [4]:
cbl_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.COBOL, chunk_size=50, chunk_overlap=0
)
cbl_docs = python_splitter.create_documents([COBOL_CODE])
cbl_docs

[Document(metadata={}, page_content='*************************************************'),
 Document(metadata={}, page_content='*****************'),
 Document(metadata={}, page_content='* Created: Thu, 14 Mar 2024 23:37:26 GMT'),
 Document(metadata={}, page_content='* Generated by: IBM watsonx Code Assistant'),
 Document(metadata={}, page_content='for Z Refactoring'),
 Document(metadata={}, page_content='* Assistant\n      * Workbook name: LGICDB01'),
 Document(metadata={}, page_content='* Workbook id:'),
 Document(metadata={}, page_content='749a805c-1f53-4f65-b1a1-be464cca3935'),
 Document(metadata={}, page_content='* Project:'),
 Document(metadata={}, page_content='$GenApp_9c635951-ff98-4c2e-b721-5163f063503d'),
 Document(metadata={}, page_content='*************************************************'),
 Document(metadata={}, page_content='*****************'),
 Document(metadata={}, page_content='IDENTIFICATION DIVISION.'),
 Document(metadata={}, page_content='PROGRAM-ID. INSCUST.'),
 Do

In [5]:
!%pip install -qU langchain_community

from langchain_community.document_loaders.parsers.language.code_segmenter import (
    CodeSegmenter)


[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
#from langchain_community.document_loaders.parsers.language.code_segmenter import  CobolSegmenter
from langchain_community.document_loaders.parsers.language.cobol import CobolSegmenter
segmenter = CobolSegmenter(COBOL_CODE)
#len(segmenter.source_lines)
segments=segmenter.extract_functions_classes()




In [12]:
segments

["       INSERT-CUSTOMER.\n      *================================================================*\n      * Insert row into Customer table based on customer number        *\n      *================================================================*\n           MOVE ' INSERT CUSTOMER' TO EM-SQLREQ\n      *================================================================*\n           IF LGAC-NCS = 'ON'\n             EXEC SQL\n               INSERT INTO CUSTOMER\n                         ( CUSTOMERNUMBER,\n                           FIRSTNAME,\n                           LASTNAME,\n                           DATEOFBIRTH,\n                           HOUSENAME,\n                           HOUSENUMBER,\n                           POSTCODE,\n                           PHONEMOBILE,\n                           PHONEHOME,\n                           EMAILADDRESS )\n                  VALUES ( :DB2-CUSTOMERNUM-INT,\n                           :CA-FIRST-NAME,\n                           :CA-LAST-NAME

In [13]:
len(segments)

7

In [16]:
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import LanguageParser
loader = GenericLoader.from_filesystem(
                "./Test3.cbl",
                glob="*/*",
                suffixes=[".cbl"],
                parser=LanguageParser(language="cobol", parser_threshold=100),
            
            )
docs=loader.load()
#docs=loader.load_and_split()
#print(len(docs)) ##8
for document in docs:
    print("doc ", document)


doc  page_content='       INSERT-CUSTOMER.
      *================================================================*
      * Insert row into Customer table based on customer number        *
      *================================================================*
           MOVE ' INSERT CUSTOMER' TO EM-SQLREQ
      *================================================================*
           IF LGAC-NCS = 'ON'
             EXEC SQL
               INSERT INTO CUSTOMER
                         ( CUSTOMERNUMBER,
                           FIRSTNAME,
                           LASTNAME,
                           DATEOFBIRTH,
                           HOUSENAME,
                           HOUSENUMBER,
                           POSTCODE,
                           PHONEMOBILE,
                           PHONEHOME,
                           EMAILADDRESS )
                  VALUES ( :DB2-CUSTOMERNUM-INT,
                           :CA-FIRST-NAME,
                           :CA-LAST-NAME,
  

In [111]:
from langchain_text_splitters import (
    Language,
    RecursiveCharacterTextSplitter,
)

In [112]:
js_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.COBOL, chunk_size=60, chunk_overlap=0
)

In [113]:
result = js_splitter.split_documents(docs)

In [115]:
len(result)

112

In [117]:
print("\n\n--8<--\n\n".join([document.page_content for document in result]))

INSERT-CUSTOMER.

--8<--

*==========================================================

--8<--

======*

--8<--

* Insert row into Customer table based on customer

--8<--

number        *

--8<--

*==========================================================

--8<--

======*

--8<--

MOVE ' INSERT CUSTOMER' TO EM-SQLREQ

--8<--

*==========================================================

--8<--

======*

--8<--

IF LGAC-NCS = 'ON'
             EXEC SQL

--8<--

INSERT INTO CUSTOMER

--8<--

( CUSTOMERNUMBER,

--8<--

FIRSTNAME,

--8<--

LASTNAME,

--8<--

DATEOFBIRTH,

--8<--

HOUSENAME,

--8<--

HOUSENUMBER,

--8<--

POSTCODE,

--8<--

PHONEMOBILE,

--8<--

PHONEHOME,

--8<--

EMAILADDRESS )

--8<--

VALUES ( :DB2-CUSTOMERNUM-INT,

--8<--

:CA-FIRST-NAME,

--8<--

:CA-LAST-NAME,

--8<--

:CA-DOB,

--8<--

:CA-HOUSE-NAME,

--8<--

:CA-HOUSE-NUM,

--8<--

:CA-POSTCODE,

--8<--

:CA-PHONE-MOBILE,

--8<--

:CA-PHONE-HOME,

--8<--

:CA-EMAIL-ADDRESS )

--8<--

END-EXEC
             IF SQLCO